# SDH exp_002_baseline

공용 전처리 벤치마크의 기준 실험입니다.

- 전처리: 결측값 → `WT`, `WT=0`, 변이=1
- 검증: `StratifiedKFold-5`, shuffle, seed=42
- 모델: 공용 고정 Logistic Regression
- 주 지표: 전체 OOF Macro F1

전처리는 공용 sklearn `Pipeline`, fold·모델·평가는 `common.preprocessing_benchmark`에서 가져옵니다.

In [ ]:
from pathlib import Path
import sys

import pandas as pd
import yaml
from IPython.display import display


def find_project_root(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / "configs" / "baseline.yaml").exists():
            return path
    raise FileNotFoundError(
        "configs/baseline.yaml을 기준으로 저장소 루트를 찾지 못했습니다."
    )


ROOT = find_project_root(Path.cwd())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

with (ROOT / "configs" / "baseline.yaml").open(encoding="utf-8") as file:
    data_config = yaml.safe_load(file)["data"]

TARGET = data_config["target_column"]
ID_COLUMN = data_config["id_column"]
EXPERIMENT_ID = "sdh-exp-002-baseline"
RESULT_DIR = ROOT / "experiments" / "SDH" / "exp_002_baseline" / "results"

print("project root:", ROOT)
print("data config:", data_config)

In [ ]:
train = pd.read_csv(ROOT / data_config["train_path"], low_memory=False)

print("train:", train.shape)
print("classes:", train[TARGET].nunique())
display(train[[ID_COLUMN, TARGET]].head())

## Baseline 전처리

`make_baseline_preprocessor()`가 WT 이진화 sklearn `Pipeline`을 생성합니다.

In [ ]:
from common.preprocessing_benchmark import run_preprocessing_benchmark
from common.starter_preprocess import make_baseline_preprocessor

baseline_preprocessor = make_baseline_preprocessor()
baseline_preprocessor

## 1차 공용 벤치마크

고정된 5-fold와 Logistic Regression으로 OOF 성능을 계산합니다. 전처리의 `fit`은 각 fold의 학습 부분에만 호출됩니다.

In [ ]:
baseline_result = run_preprocessing_benchmark(
    train,
    baseline_preprocessor,
    experiment_id=EXPERIMENT_ID,
    preprocessing_name="WT/variant binary",
)

display(baseline_result.summary_frame())

In [ ]:
display(
    baseline_result.fold_metrics[
        [
            "cv_seed",
            "fold",
            "train_rows",
            "valid_rows",
            "feature_count",
            "accuracy",
            "f1_macro",
            "elapsed_seconds",
        ]
    ]
)

display(baseline_result.oof_predictions.head())
print("OOF rows:", len(baseline_result.oof_predictions))
print("unique OOF IDs:", baseline_result.oof_predictions[ID_COLUMN].nunique())

## 결과 저장

`metrics.json`에는 경량 지표만 저장합니다. OOF 예측과 확률은 Git에 커밋하지 않습니다.

In [ ]:
metrics_path = baseline_result.save_metrics(RESULT_DIR / "metrics.json")
print("saved:", metrics_path)

## 선택 실행

아래 셀은 기본 실행 범위가 아닙니다.

- LightGBM seed 42: `metrics_lightgbm_seed42.json`
- Logistic seed 42/52/62: `metrics_logistic_seeds_42_52_62.json`

필요할 때 주석을 해제해서 실행합니다.

In [ ]:
lgbm_result = run_preprocessing_benchmark(
    train,
    baseline_preprocessor,
    experiment_id=EXPERIMENT_ID,
    preprocessing_name="WT/variant binary",
    model="lightgbm",
)
lgbm_metrics_path = lgbm_result.save_metrics(
    RESULT_DIR / "metrics_lightgbm_seed42.json"
)
display(lgbm_result.summary_frame())
display(lgbm_result.fold_metrics)
print("saved:", lgbm_metrics_path)

In [ ]:
confirmed_result = run_preprocessing_benchmark(
    train,
    baseline_preprocessor,
    experiment_id=EXPERIMENT_ID,
    preprocessing_name="WT/variant binary",
    confirmation=True,
    model="lightgbm",
)
confirmed_metrics_path = confirmed_result.save_metrics(
    RESULT_DIR / "metrics_lightgbm_seeds_42_52_62.json"
)
display(confirmed_result.summary_frame())
display(confirmed_result.run_metrics)
print("saved:", confirmed_metrics_path)

## LB 제출 파일 생성

공용 baseline과 동일한 WT 이진화 및 Logistic Regression을 전체 train에 학습하고 test를 예측합니다.

저장 파일: `results/submission_logistic_baseline.csv`

In [ ]:
import importlib

import common.starter_preprocess as starter_preprocess
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

# common 모듈을 수정한 뒤에도 커널 재시작 없이 최신 코드를 사용합니다.
starter_preprocess = importlib.reload(starter_preprocess)

test = pd.read_csv(ROOT / data_config["test_path"], low_memory=False)
submission = pd.read_csv(
    ROOT / data_config["submission_path"],
    low_memory=False,
)

feature_columns = [
    column
    for column in train.columns
    if column not in {ID_COLUMN, TARGET}
]
test_feature_columns = [
    column
    for column in test.columns
    if column != ID_COLUMN
]

if feature_columns != test_feature_columns:
    raise ValueError("train/test feature 컬럼 또는 순서가 다릅니다.")
if len(test) != len(submission):
    raise ValueError("test와 sample_submission의 행 수가 다릅니다.")
if not test[ID_COLUMN].equals(submission[ID_COLUMN]):
    raise ValueError("test와 sample_submission의 ID 또는 순서가 다릅니다.")

submission_model = Pipeline(
    [
        ("preprocessing", starter_preprocess.make_baseline_preprocessor()),
        (
            "model",
            LogisticRegression(
                solver="lbfgs",
                max_iter=1000,
                class_weight="balanced",
                random_state=42,
            ),
        ),
    ]
)

submission_model.fit(
    train[feature_columns],
    train[TARGET],
)
test_prediction = submission_model.predict(test[feature_columns])

submission[TARGET] = test_prediction
submission_path = RESULT_DIR / "submission_logistic_baseline.csv"
submission.to_csv(submission_path, index=False)

train_classes = set(train[TARGET].unique())
predicted_classes = set(test_prediction)
if not predicted_classes <= train_classes:
    raise ValueError("학습 데이터에 없는 클래스가 예측됐습니다.")
if submission.columns.tolist() != [ID_COLUMN, TARGET]:
    raise ValueError("제출 파일의 컬럼 형식이 올바르지 않습니다.")
if not submission[ID_COLUMN].is_unique:
    raise ValueError("제출 파일에 중복 ID가 있습니다.")
if submission[TARGET].isna().any():
    raise ValueError("제출 예측에 결측값이 있습니다.")

print("saved:", submission_path)
print("shape:", submission.shape)
print("predicted classes:", len(predicted_classes), "/", len(train_classes))
display(submission.head())
display(submission[TARGET].value_counts().rename("count").to_frame())

## 5-Fold OOF 모델·전처리·Soft Voting Sprint

각 후보는 동일한 Stratified 5-Fold에서 OOF와 test 확률을 동시에 생성합니다.

- `fast`: 당일 제출용 핵심 후보 약 13개
- `full`: LR 규제, class weight, 전처리 및 모델 후보 추가
- 전처리 후보: WT 이진화, 상수열 제거, 최소 변이 빈도, Chi² 선택, TF-IDF 가중
- 결과: 최고 단일 모델, 최고 soft voting, 상위 3개 균등 앙상블 제출 파일

먼저 `fast`로 실행하고 시간이 남을 때만 `full`로 변경합니다.

In [ ]:
import importlib

import experiments.SDH.exp_002_baseline.oof_ensemble as oof_ensemble

oof_ensemble = importlib.reload(oof_ensemble)

test = pd.read_csv(ROOT / data_config["test_path"], low_memory=False)
sample_submission = pd.read_csv(
    ROOT / data_config["submission_path"],
    low_memory=False,
)

ENSEMBLE_MODE = "full"  # 시간이 충분하면 "full"

ensemble_result = oof_ensemble.run_oof_ensemble_sprint(
    train=train,
    test=test,
    sample_submission=sample_submission,
    output_dir=RESULT_DIR,
    mode=ENSEMBLE_MODE,
    target_column=TARGET,
    id_column=ID_COLUMN,
)

display(ensemble_result.leaderboard)
display(ensemble_result.blend_leaderboard.head(10))
ensemble_result.submission_paths